In [2]:
import pandas as pd

In [8]:


train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")
sample = pd.read_csv("../data/sample_submission.csv")

print("train shape: ", train.shape)
print("test shape: ", test.shape)

train shape:  (577347, 12)
test shape:  (247435, 11)


In [4]:
train.head()

,id,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,class
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,GALAXY
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,GALAXY
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,QSO
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,GALAXY
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,GALAXY


In [7]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 577347 entries, 0 to 577346
Data columns (total 12 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   id                 577347 non-null  int64  
 1   alpha              577347 non-null  float64
 2   delta              577347 non-null  float64
 3   u                  577347 non-null  float64
 4   g                  577347 non-null  float64
 5   r                  577347 non-null  float64
 6   i                  577347 non-null  float64
 7   z                  577347 non-null  float64
 8   redshift           577347 non-null  float64
 9   spectral_type      577347 non-null  str    
 10  galaxy_population  577347 non-null  str    
 11  class              577347 non-null  str    
dtypes: float64(8), int64(1), str(3)
memory usage: 52.9 MB


In [8]:
train.describe()

,id,alpha,delta,u,g,r,i,z,redshift
count,577347.00000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000,577347.000000
mean,288673.00000,181.616673,21.834654,22.441926,21.007273,19.962811,19.378911,19.041136,0.723135
std,166665.86727,96.242941,18.933570,2.018135,1.795426,1.648964,1.580059,1.584365,0.810070
min,0.00000,0.011684,-17.966988,-0.139225,13.535483,12.579407,11.962781,11.682803,-0.009970
25%,144336.50000,132.161499,2.474097,20.977090,19.865005,18.820671,18.306820,17.973192,0.181052
50%,288673.00000,188.681465,21.484412,22.570222,21.467820,20.431153,19.631642,19.188598,0.497525
75%,433009.50000,231.829693,36.988310,23.869103,22.292715,21.164096,20.608191,20.162111,0.881390
max,577346.00000,359.999810,79.158322,28.253263,27.620208,25.254499,27.910853,26.826867,7.010780


In [9]:
train["class"].value_counts()

class
GALAXY    377480
QSO       117143
STAR       82724
Name: count, dtype: int64

In [10]:
train["spectral_type"].value_counts()

spectral_type
M      303323
A/F    122122
G/K    108546
O/B     43356
Name: count, dtype: int64

In [11]:
train["galaxy_population"].value_counts()

galaxy_population
Red_Sequence    319565
Blue_Cloud      257782
Name: count, dtype: int64

In [12]:
pd.crosstab(train["spectral_type"], train["class"])

class,GALAXY,QSO,STAR
spectral_type,,,
A/F,24240,61514,36368
G/K,61627,20921,25998
M,288023,3889,11411
O/B,3590,30819,8947


In [13]:
pd.crosstab(train["galaxy_population"], train["class"])

class,GALAXY,QSO,STAR
galaxy_population,,,
Blue_Cloud,88962,108274,60546
Red_Sequence,288518,8869,22178


In [11]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
import numpy as np

In [9]:
X = train.drop(columns=["id", "class"])
y = train["class"]

In [16]:
cat_features = ["spectral_type", "galaxy_population"]

In [17]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

In [18]:
for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.1,
        depth=6,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_features
    )

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(
        y_valid,
        preds
    )

    scores.append(score)

    print(score)

0.9463923117967173
0.9485192443437427
0.9482826712058223
0.9472934018201533
0.9467292033201962


In [19]:
print("Mean CV Score:", np.mean(scores))

Mean CV Score: 0.9474433664973263


#### Train on the full dataset:

In [27]:
model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    verbose=1
)

model.fit(
    X,
    y,
    cat_features=["spectral_type", "galaxy_population"]
)

0:	learn: 0.9478498	total: 578ms	remaining: 4m 48s
1:	learn: 0.8349083	total: 993ms	remaining: 4m 7s
2:	learn: 0.7453989	total: 1.26s	remaining: 3m 28s
3:	learn: 0.6726046	total: 1.56s	remaining: 3m 14s
4:	learn: 0.6128649	total: 1.85s	remaining: 3m 3s
5:	learn: 0.5616140	total: 2.13s	remaining: 2m 55s
6:	learn: 0.5178888	total: 2.4s	remaining: 2m 49s
7:	learn: 0.4804221	total: 2.71s	remaining: 2m 46s
8:	learn: 0.4472268	total: 3.03s	remaining: 2m 45s
9:	learn: 0.4192323	total: 3.33s	remaining: 2m 43s
10:	learn: 0.3939768	total: 3.63s	remaining: 2m 41s
11:	learn: 0.3720003	total: 3.91s	remaining: 2m 38s
12:	learn: 0.3512100	total: 4.22s	remaining: 2m 38s
13:	learn: 0.3325782	total: 4.48s	remaining: 2m 35s
14:	learn: 0.3159990	total: 4.75s	remaining: 2m 33s
15:	learn: 0.3013850	total: 5.03s	remaining: 2m 32s
16:	learn: 0.2887640	total: 5.28s	remaining: 2m 29s
17:	learn: 0.2752255	total: 5.53s	remaining: 2m 28s
18:	learn: 0.2643134	total: 5.8s	remaining: 2m 26s
19:	learn: 0.2542665	total

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.1, verbose=1)

In [28]:
model.save_model("../models/catboost_v1.cbm")

In [29]:
X_test = test.drop(columns=["id"])

preds = model.predict(X_test)

In [30]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": preds.flatten()
})

submission.to_csv(
    "../submissions/submission_v1.csv",
    index=False
)

In [5]:
from catboost import CatBoostClassifier
model = CatBoostClassifier()
model.load_model("../models/catboost_v1.cbm")

CatBoostClassifier(depth=6, iterations=500, learning_rate=0.1, loss_function='MultiClass', verbose=1)

In [31]:
submission.head()

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


### Experiment v2

In [10]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.get_feature_importance()
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
7,redshift,34.565155
6,z,12.282269
2,u,9.544425
3,g,9.195414
0,alpha,8.375548
1,delta,6.393684
8,spectral_type,6.167067
9,galaxy_population,4.873580
5,i,4.432424
4,r,4.170433


# ==================================
# Experiment V2
# Stronger CatBoost
# ==================================

In [12]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X, y):

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        verbose=1
    )

    model.fit(
        X_train,
        y_train,
        cat_features=["spectral_type", "galaxy_population"]
    )

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, preds)

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

0:	learn: 1.0196973	total: 502ms	remaining: 8m 21s
1:	learn: 0.9513594	total: 858ms	remaining: 7m 8s
2:	learn: 0.8909209	total: 1.21s	remaining: 6m 42s
3:	learn: 0.8377900	total: 1.57s	remaining: 6m 30s
4:	learn: 0.7899593	total: 1.91s	remaining: 6m 19s
5:	learn: 0.7469625	total: 2.25s	remaining: 6m 12s
6:	learn: 0.7078803	total: 2.59s	remaining: 6m 7s
7:	learn: 0.6725852	total: 2.94s	remaining: 6m 4s
8:	learn: 0.6403509	total: 3.28s	remaining: 6m 1s
9:	learn: 0.6105671	total: 3.62s	remaining: 5m 58s
10:	learn: 0.5834226	total: 3.97s	remaining: 5m 56s
11:	learn: 0.5582966	total: 4.32s	remaining: 5m 55s
12:	learn: 0.5349597	total: 4.65s	remaining: 5m 53s
13:	learn: 0.5133974	total: 4.99s	remaining: 5m 51s
14:	learn: 0.4928081	total: 5.35s	remaining: 5m 51s
15:	learn: 0.4740191	total: 5.68s	remaining: 5m 49s
16:	learn: 0.4564087	total: 6.03s	remaining: 5m 48s
17:	learn: 0.4398333	total: 6.38s	remaining: 5m 47s
18:	learn: 0.4245064	total: 6.73s	remaining: 5m 47s
19:	learn: 0.4097632	total

In [13]:
model_v2 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

model_v2.fit(
    X,
    y,
    cat_features=["spectral_type", "galaxy_population"]
)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.05, verbose=0)

In [14]:
model_v2.save_model("../models/catboost_v2.cbm")

In [15]:
X_test = test.drop(columns=["id"])

preds = model_v2.predict(X_test)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds.flatten()
})

submission.to_csv(
    "../submissions/submission_v2.csv",
    index=False
)

# ==================================
# Experiment V3
# Feature Engineering
# ==================================

In [16]:
X_fe = X.copy()
X_test_fe = test.drop(columns=["id"]).copy()

In [17]:
X_fe["u_g"] = X_fe["u"] - X_fe["g"]
X_fe["g_r"] = X_fe["g"] - X_fe["r"]
X_fe["r_i"] = X_fe["r"] - X_fe["i"]
X_fe["i_z"] = X_fe["i"] - X_fe["z"]

In [18]:
X_test_fe["u_g"] = X_test_fe["u"] - X_test_fe["g"]
X_test_fe["g_r"] = X_test_fe["g"] - X_test_fe["r"]
X_test_fe["r_i"] = X_test_fe["r"] - X_test_fe["i"]
X_test_fe["i_z"] = X_test_fe["i"] - X_test_fe["z"]

In [19]:
print(X_fe.columns)

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z'],
      dtype='str')


In [22]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X_fe, y):

    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        cat_features=["spectral_type", "galaxy_population"]
    )

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, preds)

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.949647
Fold Score: 0.951436
Fold Score: 0.949838
Fold Score: 0.950853
Fold Score: 0.950572

Mean CV Score: 0.9504691423379971


In [23]:
model_v3 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

model_v3.fit(
    X_fe,
    y,
    cat_features=["spectral_type", "galaxy_population"]
)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.05, verbose=0)

In [24]:
preds = model_v3.predict(X_test_fe)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds.flatten()
})

submission.to_csv(
    "../submissions/submission_v3.csv",
    index=False
)

In [25]:
model_v3.save_model("../models/catboost_v3.cbm")

### Next Experiment (V4)

let's add a few more astronomy-inspired features.

In [26]:
X_fe["u_r"] = X_fe["u"] - X_fe["r"]
X_fe["g_i"] = X_fe["g"] - X_fe["i"]
X_fe["r_z"] = X_fe["r"] - X_fe["z"]

In [27]:
X_test_fe["u_r"] = X_test_fe["u"] - X_test_fe["r"]
X_test_fe["g_i"] = X_test_fe["g"] - X_test_fe["i"]
X_test_fe["r_z"] = X_test_fe["r"] - X_test_fe["z"]

In [28]:
print(X_fe.columns)

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type',
       'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'r_z'],
      dtype='str')


In [29]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X_fe, y):

    X_train = X_fe.iloc[train_idx]
    X_valid = X_fe.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        cat_features=["spectral_type", "galaxy_population"]
    )

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, preds)

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.950009
Fold Score: 0.951617
Fold Score: 0.950912
Fold Score: 0.951151
Fold Score: 0.951626

Mean CV Score: 0.9510631811088202


In [30]:
model_v4 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

model_v4.fit(
    X_fe,
    y,
    cat_features=["spectral_type", "galaxy_population"]
)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.05, verbose=0)

In [31]:
model_v4.save_model("../models/catboost_v4.cbm")

In [32]:
preds = model_v4.predict(X_test_fe)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds.flatten()
})

submission.to_csv(
    "../submissions/submission_v4.csv",
    index=False
)

### Experiment V5: Feature Importance on V4

In [33]:
feature_importance = pd.DataFrame({
    "Feature": X_fe.columns,
    "Importance": model_v4.get_feature_importance()
})

feature_importance.sort_values(
    by="Importance",
    ascending=False
)

,Feature,Importance
7,redshift,34.785292
0,alpha,10.523213
15,g_i,8.789111
1,delta,8.481540
6,z,5.996740
16,r_z,5.134568
14,u_r,4.046187
10,u_g,3.883111
3,g,3.423199
2,u,3.408275


## Surprising Discovery #1

Look at:

spectral_type    :   0.59

galaxy_population :  0.27

These are almost useless now.

Initially they seemed important, but after feature engineering CatBoost is extracting the same information from the numerical features.

This is a good example of why we check feature importance instead of making assumptions.

## Surprising Discovery #2

Your engineered features are working!

g_i  = 8.79

r_z  = 5.13

u_r  = 4.05

u_g  = 3.88

Some of these are more important than the original magnitude features.

That's exactly why your leaderboard score increased.

# Experiment V5 (High Value)

Let's test whether those two categorical columns are still needed.

In [34]:
X_v5 = X_fe.drop(
    columns=["spectral_type", "galaxy_population"]
)

X_test_v5 = X_test_fe.drop(
    columns=["spectral_type", "galaxy_population"]
)

In [36]:
print(X_v5.columns)

Index(['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'u_g', 'g_r',
       'r_i', 'i_z', 'u_r', 'g_i', 'r_z'],
      dtype='str')


In [38]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = []

for train_idx, valid_idx in skf.split(X_v5, y):

    X_train = X_v5.iloc[train_idx]
    X_valid = X_v5.iloc[valid_idx]

    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        # cat_features=["spectral_type", "galaxy_population"]
    )

    preds = model.predict(X_valid)

    score = balanced_accuracy_score(y_valid, preds)

    scores.append(score)

    print(f"Fold Score: {score:.6f}")

print("\nMean CV Score:", np.mean(scores))

Fold Score: 0.949848
Fold Score: 0.952363
Fold Score: 0.950895
Fold Score: 0.951659
Fold Score: 0.951220

Mean CV Score: 0.9511968978771405


In [40]:
model_v5 = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    verbose=0
)

model_v5.fit(
    X_v5,
    y,
    # cat_features=["spectral_type", "galaxy_population"]
)

CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.05, verbose=0)

In [41]:
model_v5.save_model("../models/catboost_v5.cbm")

In [42]:
preds = model_v5.predict(X_test_v5)

submission = pd.DataFrame({
    "id": test["id"],
    "class": preds.flatten()
})

submission.to_csv(
    "../submissions/submission_v5.csv",
    index=False
)